# Notebook 04: Complete Agentic Chatbot

Adding conversation memory and user context extraction.

## 1. Setup

In [15]:
import os
import json
import re
import random
from typing import Dict, List
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import boto3

load_dotenv()

LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')
KNOWLEDGE_BASE_ID = os.getenv('KNOWLEDGE_BASE_ID')

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

print('[OK] Configuration loaded')

[OK] Configuration loaded


## 2. Tools and Schemas

In [16]:
# Mock data
MOCK_ORDERS = {
    "ORD-12345": {"status": "shipped", "carrier": "FedEx", "estimated_delivery": "2026-02-03", 
                  "items": ["Blue iPhone 15 Case"], "destination": "New York, NY"},
    "ORD-67890": {"status": "processing", "estimated_delivery": "2026-02-05", 
                  "items": ["Wireless Earbuds"], "destination": "Miami, FL"},
    "ORD-11111": {"status": "delivered", "carrier": "UPS", "delivered_date": "2026-01-28", 
                  "items": ["Laptop Stand"], "destination": "Los Angeles, CA"}
}

MOCK_WEATHER = {
    "miami": {"location": "Miami, FL", "condition": "Hurricane Warning", "estimated_delay_days": 3},
    "new york": {"location": "New York, NY", "condition": "Clear", "estimated_delay_days": 0}
}

MOCK_INVENTORY = {
    "iphone 15 case": {"blue": {"in_stock": True, "quantity": 42}, "black": {"in_stock": False, "quantity": 0}},
    "airpods pro": {"default": {"in_stock": True, "quantity": 120}}
}

# Tool implementations
def search_knowledge_base(query: str) -> Dict:
    try:
        response = bedrock_agent.retrieve(
            knowledgeBaseId=KNOWLEDGE_BASE_ID, retrievalQuery={'text': query},
            retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
        )
        return {'success': True, 'results': [{'content': r['content']['text']} for r in response.get('retrievalResults', [])]}
    except Exception as e:
        return {'success': False, 'error': str(e)}

def check_order_status(order_id: str) -> Dict:
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"): order_id = f"ORD-{order_id}"
    if order_id in MOCK_ORDERS:
        return {'success': True, 'order_id': order_id, **MOCK_ORDERS[order_id]}
    return {'success': False, 'error': f'Order {order_id} not found'}

def get_weather_alerts(location: str) -> Dict:
    key = location.lower().split(',')[0].strip()
    if key in MOCK_WEATHER:
        return {'success': True, **MOCK_WEATHER[key]}
    return {'success': True, 'location': location, 'condition': 'Clear', 'estimated_delay_days': 0}

def check_inventory(product_name: str, color: str = None) -> Dict:
    key = product_name.lower().strip()
    for pkey in MOCK_INVENTORY:
        if pkey in key or key in pkey:
            data = MOCK_INVENTORY[pkey]
            if color and color.lower() in data:
                return {'success': True, 'product': pkey.title(), **data[color.lower()]}
            return {'success': True, 'product': pkey.title(), 'variants': data}
    return {'success': False, 'error': 'Product not found'}

def create_return_request(order_id: str, reason: str) -> Dict:
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"): order_id = f"ORD-{order_id}"
    if order_id not in MOCK_ORDERS:
        return {'success': False, 'error': 'Order not found'}
    return_id = f"RET-{random.randint(10000, 99999)}"
    return {'success': True, 'return_id': return_id, 'order_id': order_id, 'label_url': f'https://returns.example.com/{return_id}'}

TOOLS = [
    {"type": "function", "function": {"name": "search_knowledge_base", "description": "Search company knowledge base for policies, FAQs, shipping info, and general questions.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "The search query"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "check_order_status", "description": "Check order status, tracking, and delivery ETA. Requires order ID.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "Order ID (e.g., ORD-12345)"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "get_weather_alerts", "description": "Check weather alerts and shipping delays for a location. Use when customer asks about weather impact on deliveries or delays to a city.",
        "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "City name (e.g., Miami, New York)"}}, "required": ["location"]}}},
    {"type": "function", "function": {"name": "check_inventory", "description": "Check if a product is in stock and available quantities.",
        "parameters": {"type": "object", "properties": {"product_name": {"type": "string", "description": "Product name"}, "color": {"type": "string", "description": "Color variant (optional)"}}, "required": ["product_name"]}}},
    {"type": "function", "function": {"name": "create_return_request", "description": "Create a return request for an order. Customer must provide order ID and reason.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "Order ID to return"}, "reason": {"type": "string", "description": "Reason for return"}}, "required": ["order_id", "reason"]}}}
]

TOOL_FUNCTIONS = {
    "search_knowledge_base": search_knowledge_base, "check_order_status": check_order_status,
    "get_weather_alerts": get_weather_alerts, "check_inventory": check_inventory,
    "create_return_request": create_return_request
}

print('[OK] Tools defined')

[OK] Tools defined


## 3. Agentic Chatbot with Memory

In [17]:
class AgenticChatbot:
    """Agentic RAG Chatbot with conversation memory and user context."""
    
    def __init__(self, model: str = None, max_iterations: int = 5, max_history: int = 10):
        self.model = model or LLM_MODEL
        self.max_iterations = max_iterations
        self.max_history = max_history
        self.conversation_history = []
        self.user_context = {}  # Extracted user info (name, orders)
        
        self.base_system_prompt = """You are a helpful e-commerce customer service agent.

Available tools:
- search_knowledge_base: For policies, FAQs, general questions
- check_order_status: For order tracking (requires order ID)
- get_weather_alerts: For weather-related shipping delays (just needs city name)
- check_inventory: For stock availability
- create_return_request: To process returns (requires order ID and reason)

IMPORTANT: Be proactive - use tools immediately when relevant. For weather/delay questions about a city, call get_weather_alerts directly with the city name. Don't ask for order ID unless the tool specifically requires it."""
    
    def _get_system_prompt(self) -> str:
        prompt = self.base_system_prompt
        if self.user_context:
            prompt += "\n\nUser context: " + str(self.user_context)
        return prompt
    
    def _extract_user_context(self, user_message: str):
        # Extract name
        for pattern in [r"my name is (\w+)", r"i'm (\w+)", r"i am (\w+)"]:
            match = re.search(pattern, user_message.lower())
            if match:
                self.user_context['name'] = match.group(1).capitalize()
                break
        # Extract order ID
        order_match = re.search(r'ord-?(\d+)', user_message.lower())
        if order_match:
            self.user_context['last_order'] = f"ORD-{order_match.group(1)}"
    
    def _execute_tool(self, name: str, args: Dict) -> str:
        result = TOOL_FUNCTIONS[name](**args) if name in TOOL_FUNCTIONS else {"error": "Unknown tool"}
        return json.dumps(result)
    
    def chat(self, user_message: str) -> Dict:
        self._extract_user_context(user_message)
        
        # Build messages with history
        messages = [{"role": "system", "content": self._get_system_prompt()}]
        for h in self.conversation_history[-self.max_history:]:
            messages.append({"role": "user", "content": h['user']})
            messages.append({"role": "assistant", "content": h['assistant']})
        messages.append({"role": "user", "content": user_message})
        
        tools_used = []
        
        for _ in range(self.max_iterations):
            response = llm_client.chat.completions.create(
                model=self.model, messages=messages, tools=TOOLS, tool_choice="auto"
            )
            msg = response.choices[0].message
            
            if msg.tool_calls:
                messages.append({"role": "assistant", "content": msg.content,
                    "tool_calls": [{"id": tc.id, "type": "function",
                                   "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                  for tc in msg.tool_calls]})
                for tc in msg.tool_calls:
                    result = self._execute_tool(tc.function.name, json.loads(tc.function.arguments))
                    tools_used.append(tc.function.name)
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            else:
                # Save to history
                self.conversation_history.append({'user': user_message, 'assistant': msg.content})
                return {"response": msg.content, "tools_used": tools_used}
        
        return {"response": "Max iterations reached", "tools_used": tools_used}
    
    def clear(self):
        self.conversation_history = []
        self.user_context = {}

chatbot = AgenticChatbot()
print('[OK] Agentic Chatbot created')

[OK] Agentic Chatbot created


## 4. Demo: Multi-Turn Conversation

In [18]:
def demo_conversation(messages: List[str]):
    chatbot = AgenticChatbot()
    print("=" * 60)
    print("MULTI-TURN CONVERSATION DEMO")
    print("=" * 60)
    
    for i, msg in enumerate(messages, 1):
        print(f"\n[Turn {i}]")
        print(f"User: {msg}")
        result = chatbot.chat(msg)
        print(f"Bot: {result['response']}")
        if result['tools_used']:
            print(f"  (Tools: {result['tools_used']})")
    
    print("\n" + "=" * 60)
    print("User context extracted:", chatbot.user_context)

demo_conversation([
    "Hi, my name is Sarah",
    "What's the status of my order ORD-12345?",
    "Will it be delayed because of weather?",
    "Actually, I want to return it. The color doesn't match.",
    "Is the black iPhone 15 case available instead?"
])

MULTI-TURN CONVERSATION DEMO

[Turn 1]
User: Hi, my name is Sarah
Bot: Hello Sarah! How can I assist you today?

[Turn 2]
User: What's the status of my order ORD-12345?
Bot: Your order (ORD-12345) has been shipped via FedEx. It contains a Blue iPhone 15 Case and is estimated to be delivered on February 3, 2026, to New York, NY. If you have any more questions or need further assistance, feel free to ask!
  (Tools: ['check_order_status'])

[Turn 3]
User: Will it be delayed because of weather?
Bot: There are no weather-related delays in New York right now. Your order should arrive on time on February 3, 2026. If you have any more questions, feel free to ask!
  (Tools: ['get_weather_alerts'])

[Turn 4]
User: Actually, I want to return it. The color doesn't match.
Bot: I can help you with the return process for your order. Let me create a return request for you.

Could you please confirm the reason for the return as "the color doesn't match"?

[Turn 5]
User: Is the black iPhone 15 case avai

## 5. Demo: Complex Query

In [19]:
demo_conversation([
    "I have an order going to Miami. Order number is 67890. Will it arrive on time given the weather?",
    "That's frustrating. What's your policy on delays?",
    "Can I cancel and get a refund instead?"
])

MULTI-TURN CONVERSATION DEMO

[Turn 1]
User: I have an order going to Miami. Order number is 67890. Will it arrive on time given the weather?
Bot: Your order for Wireless Earbuds is currently processing and was estimated to be delivered by February 5, 2026. However, there is a Hurricane Warning in Miami, which is expected to cause an estimated delay of 3 days. Please consider this weather condition likely to impact the delivery timeline.
  (Tools: ['get_weather_alerts', 'check_order_status'])

[Turn 2]
User: That's frustrating. What's your policy on delays?
Bot: Unfortunately, I couldn't find specific information about our policy for weather-related delays. However, our general shipping delivery timeframes are as follows:

- **Standard Shipping**: Typically takes 5-7 business days from the date of shipment.
- **Express Shipping**: Available for 2-3 business day delivery.
- **International Shipments**: Usually arrive within 10-15 business days, but this can be longer for remote location

## 6. Comparison: Baseline vs Agentic

In [20]:
def baseline_rag(query: str) -> str:
    response = bedrock_agent.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID, retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
    )
    context = "\n".join([r['content']['text'] for r in response.get('retrievalResults', [])])
    llm_response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": "Answer based on context only."},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}]
    )
    return llm_response.choices[0].message.content

queries = [
    "What's the status of order ORD-12345?",
    "Is the blue iPhone 15 case in stock?",
    "Will my Miami delivery be delayed?"
]

print("COMPARISON: Baseline vs Agentic")
print("=" * 70)

for q in queries:
    chatbot_test = AgenticChatbot()
    print(f"\nQ: {q}")
    print("-" * 70)
    print(f"Baseline: {baseline_rag(q)[:150]}...")
    result = chatbot_test.chat(q)
    print(f"Agentic: {result['response'][:150]}...")
    print(f"Tools: {result['tools_used']}")

COMPARISON: Baseline vs Agentic

Q: What's the status of order ORD-12345?
----------------------------------------------------------------------
Baseline: I'm sorry, but I cannot provide the status of order ORD-12345 as the specific details of orders are not accessible in this context. Please try logging...
Agentic: Your order, **ORD-12345**, has been shipped via FedEx and is estimated to be delivered on February 3, 2026. The order includes a Blue iPhone 15 Case a...
Tools: ['check_order_status']

Q: Is the blue iPhone 15 case in stock?
----------------------------------------------------------------------
Baseline: I don't have the specific stock information for the blue iPhone 15 case. If it is not currently available, you may need to check back later or see if ...
Agentic: Yes, the blue iPhone 15 case is currently in stock with 42 units available. Would you like to order one or need further assistance?...
Tools: ['check_inventory']

Q: Will my Miami delivery be delayed?
------------

## 7. Interactive Mode (Optional)

In [14]:
def interactive_chat():
    """Interactive chat. Type 'quit' to exit, 'clear' to reset."""
    chatbot = AgenticChatbot()
    print("Agentic Chatbot (type 'quit' to exit, 'clear' to reset)")
    print("-" * 50)
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
        except KeyboardInterrupt:
            break
        
        if not user_input: continue
        if user_input.lower() == 'quit': break
        if user_input.lower() == 'clear':
            chatbot.clear()
            print("Cleared.")
            continue
        
        result = chatbot.chat(user_input)
        print(f"Bot: {result['response']}")
        if result['tools_used']:
            print(f"  (Tools: {result['tools_used']})")

# Uncomment to run:
# interactive_chat()

## Summary

Built complete Agentic Chatbot with:
- ReAct agent loop
- 5 tools (KB, orders, weather, inventory, returns)
- Conversation memory
- User context extraction

**Next:** See LangGraph implementation (Notebook 05).